# Lab 00-03: Your First LLM Call

**Module:** 00 — Foundations  
**Exam weight:** Foundational — Foundation Model APIs appear across Sections 1, 3, and 4  
**UI Sections:** Playground  
**Estimated Time:** 30–40 minutes  
**Difficulty:** Beginner

## What You Will Build

In this lab, you'll make your first call to a Large Language Model (LLM) on Databricks — both interactively through the **Playground** and programmatically from a notebook.

Databricks hosts popular open-source models (like Meta's Llama) as **Foundation Model APIs**. These are ready-to-use LLMs that you can call without downloading, configuring, or managing any infrastructure. The API is compatible with the OpenAI SDK, which means if you've ever used the OpenAI API, the code is nearly identical — you just point it at Databricks instead.

### Why This Matters for the Exam

The Foundation Model API is the backbone of every GenAI application on Databricks. The exam tests whether you know how to call it, what models are available, and what drives cost (tokens). You'll also need to know the difference between the Playground (interactive prototyping) and programmatic API calls (production use).

## Mental Model

> **Analogy:** Think of the Foundation Model API like a restaurant kitchen. You (the customer) send in an order (your prompt). The kitchen (the hosted LLM) prepares the dish (the response) and sends it back. You pay based on how much food you ordered — in LLM terms, that's **tokens** (roughly 1 token ≈ ¾ of a word). The **Playground** is like eating at the restaurant — you chat with the chef directly. The **API** is like ordering delivery — same kitchen, same food, but you get it programmatically.

### What's a token?

Tokens are the "atoms" of LLM processing. The model doesn't see words — it sees tokens.

```
"Hello, how are you?" → ["Hello", ",", " how", " are", " you", "?"] → 6 tokens
"Databricks"          → ["Data", "bricks"]                           → 2 tokens
```

You pay for **input tokens** (your prompt) + **output tokens** (the model's response). Longer prompts and longer responses cost more.

## Exam Alert

| Trap | Correct Answer |
|---|---|
| "You need to download models to use them on Databricks" | **WRONG** — Foundation Model APIs are pre-hosted and ready to call |
| "The Databricks API uses a proprietary SDK" | **WRONG** — it's OpenAI-compatible (same SDK, different base URL) |
| "Token count only includes the model's response" | **WRONG** — you pay for input tokens + output tokens |
| "The Playground is only for testing, not production" | **CORRECT** — Playground is for prototyping; use the API for production |

> **Exam tip:** When a question mentions "Foundation Model API," look for answers about the OpenAI-compatible endpoint. When it mentions "Playground," it's about interactive prototyping.

## Prerequisites

- Completed Lab 00-01 (running cluster) and Lab 00-02 (genai_labs schema exists)
- Access to Foundation Model APIs (available on most Databricks workspaces)

## Databricks UI: Using the Playground

Before writing code, let's use the Playground for an interactive conversation:

1. **UI ->** Left nav -> **Playground** (the chat bubble icon)
2. In the top-left dropdown, select a model: **databricks-meta-llama-3-3-70b-instruct**
3. Type a message: `What is Databricks Unity Catalog in one sentence?`
4. Press Enter and observe the response
5. Notice the **token count** displayed at the bottom of each message

Try a few more messages:
- `Explain RAG in simple terms`
- `Write a Python function that reverses a string`

Notice how each response shows the token count. Longer responses = more tokens = more cost.

6. Click the **Export to notebook** button (top-right) — this generates Python code you can use in a notebook. That's what we'll build manually next.

In [ ]:
# ------------------------------------------------------------------
# Setup: Install the OpenAI SDK (used to call Databricks Foundation Model APIs)
# ------------------------------------------------------------------

%pip install openai --quiet

# Restart Python to pick up the new package
dbutils.library.restartPython()

**Expected output:** Installation logs followed by `Python interpreter will be restarted.`

This is normal — proceed to the next cell.

## Step 1: Make Your First Programmatic LLM Call

Now we'll call the same model you just used in the Playground, but from Python code. This is how you'll call LLMs in every subsequent lab.

The key insight: **Databricks Foundation Model APIs are OpenAI-compatible.** You use the `openai` Python SDK, but you point it at your Databricks workspace URL instead of `api.openai.com`. Everything else — the message format, the parameters, the response structure — is the same.

In [ ]:
# ------------------------------------------------------------------
# Step 1: Make your first LLM call using the OpenAI-compatible SDK
# ------------------------------------------------------------------

from openai import OpenAI

# On Databricks, get auth from the notebook context
# (works on both classic clusters and serverless)
db_host = spark.conf.get("spark.databricks.workspaceUrl")
db_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

client = OpenAI(
    api_key=db_token,
    base_url=f"https://{db_host}/serving-endpoints"
)

# Send a message to the model — same as what you did in the Playground
response = client.chat.completions.create(
    model="databricks-meta-llama-3-3-70b-instruct",  # The model you used in Playground
    messages=[
        {"role": "system", "content": "You are a helpful assistant that gives concise answers."},
        {"role": "user", "content": "What is Databricks Unity Catalog in one sentence?"}
    ],
    max_tokens=100,   # Limit the response length (saves cost)
    temperature=0.1   # Low temperature = more deterministic/consistent responses
)

# Print the response
print("=== Model Response ===")
print(response.choices[0].message.content)

**Expected output:**
```
=== Model Response ===
Unity Catalog is Databricks' unified governance solution that provides 
centralized access control, auditing, and data lineage across all data 
and AI assets in the Databricks lakehouse platform.
```

(Your exact wording will vary, but the content should be similar.)

**What just happened:** You made your first programmatic LLM call! Let's break down the key parts:

- `spark.conf.get("spark.databricks.workspaceUrl")` — Gets your workspace URL
- `dbutils.notebook.entry_point...apiToken().get()` — Gets the auth token (works on both classic and serverless)
- `OpenAI(base_url=...)` — Points the SDK at Databricks instead of OpenAI
- `messages` — A list of messages with roles: `system` (sets behavior), `user` (your input)
- `max_tokens=100` — Caps the response length (cost control)
- `temperature=0.1` — Controls randomness (0 = deterministic, 1 = creative)
- `response.choices[0].message.content` — The model's text response

## Step 2: Understand Token Usage and Cost

Every API response includes a `usage` field that tells you how many tokens were consumed. This is critical for understanding and controlling costs.

The formula is simple:

```
Total cost = (input_tokens × input_price) + (output_tokens × output_price)
```

Input tokens are usually cheaper than output tokens. The biggest cost levers are:
1. **Prompt length** — shorter system prompts save input tokens
2. **Response length** — `max_tokens` caps output cost
3. **Number of calls** — batch processing reduces per-call overhead

In [ ]:
# ------------------------------------------------------------------
# Step 2: Inspect token usage from the response
# ------------------------------------------------------------------

# The usage object is attached to every API response
usage = response.usage

print("=== Token Usage ===")
print(f"Input tokens (your prompt):    {usage.prompt_tokens}")
print(f"Output tokens (model response): {usage.completion_tokens}")
print(f"Total tokens:                   {usage.total_tokens}")
print(f"\nCost breakdown:")
print(f"  You sent ~{usage.prompt_tokens} tokens in (system prompt + user message)")
print(f"  The model generated ~{usage.completion_tokens} tokens out")
print(f"  Longer prompts or responses = more tokens = higher cost")

**Expected output:**
```
=== Token Usage ===
Input tokens (your prompt):    34
Output tokens (model response): 38
Total tokens:                   72

Cost breakdown:
  You sent ~34 tokens in (system prompt + user message)
  The model generated ~38 tokens out
  Longer prompts or responses = more tokens = higher cost
```

(Your exact numbers will vary slightly.)

**What just happened:** You inspected the token usage of your API call. Notice that even a simple one-sentence question costs ~70 tokens total. When you build RAG applications with long context windows (thousands of tokens of retrieved documents), costs add up fast. That's why `max_tokens` and efficient prompts matter.

## Step 3: Experiment with Different Parameters

Let's see how **temperature** affects the model's output. Temperature controls randomness:

| Temperature | Behavior | Use When |
|---|---|---|
| 0.0 | Deterministic — same input always gives same output | Factual Q&A, structured output, classification |
| 0.3-0.7 | Balanced | General-purpose chat |
| 1.0+ | Creative — more varied and surprising outputs | Brainstorming, creative writing |

For most GenAI engineering tasks (RAG, classification, extraction), you want **low temperature** (0.0–0.3) for consistency.

In [ ]:
# ------------------------------------------------------------------
# Step 3: Compare outputs at different temperatures
# ------------------------------------------------------------------

prompt = "Give me a one-sentence summary of what RAG (Retrieval-Augmented Generation) does."

for temp in [0.0, 0.5, 1.0]:
    response = client.chat.completions.create(
        model="databricks-meta-llama-3-3-70b-instruct",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=temp
    )
    print(f"\n--- Temperature {temp} ---")
    print(response.choices[0].message.content.strip())

**Expected output:** Three responses to the same question, each slightly different:

```
--- Temperature 0.0 ---
RAG retrieves relevant documents from a knowledge base and feeds them into 
a language model to generate more accurate, grounded responses.

--- Temperature 0.5 ---
RAG combines information retrieval with text generation, allowing a language 
model to access external knowledge to produce more informed answers.

--- Temperature 1.0 ---
Retrieval-Augmented Generation fetches contextual snippets from a data 
repository and weaves them into the generative process for richer outputs.
```

**What just happened:** All three answers are correct, but notice how the language gets more varied at higher temperatures. At `temp=0.0`, running the same prompt twice gives the exact same output. At `temp=1.0`, each run may produce different wording.

> **For the exam:** When a question asks about "consistency" or "deterministic output," the answer is low temperature. When it asks about "creative" or "diverse" output, the answer is high temperature.

## Step 4: Explore Available Models

Databricks hosts several foundation models. The model you choose depends on your task:

| Model | Best For | Size |
|---|---|---|
| `databricks-meta-llama-3-3-70b-instruct` | General chat, reasoning, code | 70B parameters |
| `databricks-meta-llama-3-1-8b-instruct` | Lighter tasks, lower cost | 8B parameters |
| `databricks-gte-large-en` | Text embeddings (for Vector Search) | Embedding model |
| `databricks-bge-large-en` | Text embeddings (alternative) | Embedding model |

You can see all available models in two places:
- **UI ->** Left nav -> **Playground** -> model dropdown at the top
- **UI ->** Left nav -> **Serving** -> lists all active endpoints

Let's list the available serving endpoints programmatically.

In [ ]:
# ------------------------------------------------------------------
# Step 4: List available foundation model endpoints
# ------------------------------------------------------------------

import requests

# Reuse the auth from Step 1
headers = {"Authorization": f"Bearer {db_token}"}
resp = requests.get(f"https://{db_host}/api/2.0/serving-endpoints", headers=headers)

if resp.status_code == 200:
    endpoints = resp.json().get("endpoints", [])
    print(f"=== Available Serving Endpoints ({len(endpoints)} total) ===")
    for ep in endpoints[:10]:  # Show first 10
        state = ep.get("state", {}).get("ready", "UNKNOWN")
        print(f"  {ep['name']:<50} State: {state}")
else:
    print(f"Could not list endpoints (status {resp.status_code})")
    print("This is normal on Community Edition — endpoints are listed in the Serving UI.")

**Expected output:**
```
=== Available Serving Endpoints (N total) ===
  databricks-meta-llama-3-3-70b-instruct             State: READY
  databricks-meta-llama-3-1-8b-instruct              State: READY
  databricks-gte-large-en                            State: READY
  ...
```

**What just happened:** You queried the Databricks Serving API to see which models are available. The `READY` state means the model is warmed up and accepting requests. Foundation models with the `databricks-` prefix are pre-hosted by Databricks — you don't need to deploy or manage them.

## Exam Tips & Traps

**Quick-scan reference — review these before the exam:**

| # | Topic | Trap | Correct Answer |
|---|---|---|---|
| 1 | API compatibility | "Databricks uses a custom API format" | **WRONG** — it's OpenAI-compatible (same SDK, different base URL) |
| 2 | Token cost | "Cost depends only on response length" | **WRONG** — cost = input tokens + output tokens |
| 3 | Temperature | "Higher temperature = better quality" | **WRONG** — higher temperature = more random, not more accurate |
| 4 | Playground | "Export from Playground produces production-ready code" | **Partially correct** — it produces working code, but you'll add error handling, guardrails, etc. |
| 5 | Models | "You must fine-tune a model before using it" | **WRONG** — Foundation Model APIs work out of the box (zero config) |

> **Pattern to watch for:** When the exam mentions "OpenAI-compatible" or "Foundation Model API," it's referring to the pay-per-token hosted models. When it mentions "provisioned throughput," that's a different tier (covered in Lab 04-04).

## Key Takeaways

### Core Concepts
- LLMs process text as **tokens** (~¾ word each). Cost = input tokens + output tokens.
- **Temperature** controls randomness: 0.0 = deterministic, 1.0 = creative
- The message format uses **roles**: `system` (behavior), `user` (input), `assistant` (output)
- `max_tokens` caps response length and is a key cost control

### Databricks-Specific
- Foundation Model APIs are **OpenAI-compatible** — same SDK, different `base_url`
- **Playground** = interactive prototyping; **API** = programmatic/production use
- Authentication uses the notebook context (`dbutils.notebook...apiToken()`) — works on both classic and serverless
- Models with the `databricks-` prefix are pre-hosted and pay-per-token

### Exam Essentials
- Know the OpenAI-compatible pattern: `client = OpenAI(base_url=..., api_key=...)`
- Know that cost is driven by tokens (input + output)
- Know that Playground is for prototyping, API is for production
- Know the two model types: chat/instruct models (text generation) and embedding models (vector generation)

---

## Next Lab

**Lab 01-01: Prompt Engineering** — you'll learn systematic prompting techniques (zero-shot, few-shot, chain-of-thought) that make LLM outputs reliable and structured.

> **Congratulations!** You've completed Module 00 — Foundations. You now have a running cluster, a personal schema in Unity Catalog, and the ability to call foundation models. Everything from here builds on these skills.